# FaceMesh para detecção de rostos, olhos e boca

In [ ]:
import json
import cv2
import pandas as pd
import numpy as np
import dlib
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display

In [ ]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

In [ ]:
def play_video(video_path, interval=40, max_frames=120):
    frames = load_video_frames(video_path)

    if len(frames) == 0:
        raise ValueError("Nao ha frames para exibir.")

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")

    img = ax.imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))

    def update(i):
        frame = frames[i]
        img.set_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

# Metadatas

In [ ]:
df = pd.read_csv("C:\\Users\\Guilherme Monteiro\\Desktop\\TCC\\data\\video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "C:\\Users\\Guilherme Monteiro\\Desktop\\TCC\\data\\videos\\" + x)


In [ ]:
metadata_true = df[df['Video Ground Truth'] == "Real"]
metadata_fake = df[df['Video Ground Truth'] == "Fake"]

## Funcoes auxiliares

In [ ]:
def draw_text_block(img, texts, x=10, y=20):
    overlay = img.copy()

    h = 20 * len(texts) + 10
    w = 350

    # retângulo escuro
    cv2.rectangle(overlay, (x-5, y-20), (x+w, y+h), (0,0,0), -1)

    # transparência
    alpha = 0.6
    img = cv2.addWeighted(overlay, alpha, img, 1-alpha, 0)

    # texto
    for i, text in enumerate(texts):
        cv2.putText(
            img,
            text,
            (x, y + i*20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255,255,255),
            1,
            cv2.LINE_AA
        )

    return img

# Instabilidade temporal

In [ ]:
import numpy as np

def instabilidade_temporal(metrics, window=5):
    
    def extract_series(key):
        return np.array([m[key] for m in metrics if key in m])

    def moving_average(x, w):
        if len(x) < w:
            return x
        return np.convolve(x, np.ones(w)/w, mode='valid')

    def compute_temporal_stats(series):
        if len(series) < 2:
            return {
                "var_global": 0,
                "instabilidade": 0,
                "eventos": 0,
                "inst_relativa": 0
            }

        # suavização
        smooth = moving_average(series, window)

        # variabilidade global
        var_global = np.std(smooth)

        # diferenças entre frames
        diffs = np.diff(smooth)

        # instabilidade (magnitude média das mudanças)
        instabilidade = np.mean(np.abs(diffs))

        # instabilidade relativa
        inst_rel = np.mean(np.abs(diffs / (smooth[:-1] + 1e-6)))

        # eventos abruptos (threshold adaptativo)
        threshold = np.mean(np.abs(diffs)) + 2*np.std(diffs)
        eventos = np.sum(np.abs(diffs) > threshold)

        return {
            "var_global": var_global,
            "instabilidade": instabilidade,
            "eventos": eventos,
            "inst_relativa": inst_rel
        }

    metrics_temporal = {}

    # features mais relevantes
    features = [
        "num_keypoints",
        "kp_x_std",
        "kp_y_std"
    ]

    for region in ["face", "mouth"]:
        for feat in features:
            key = f"{region}_{feat}"

            series = extract_series(key)
            stats = compute_temporal_stats(series)

            for stat_name, value in stats.items():
                metrics_temporal[f"{region}_{feat}_{stat_name}"] = value

    return metrics_temporal

    

# LBP

In [ ]:
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cosine
import numpy as np
import cv2


def _normalize_to_uint8(array):
    array = np.asarray(array, dtype=np.float32)
    min_value = float(np.min(array))
    max_value = float(np.max(array))

    if max_value - min_value < 1e-6:
        return np.zeros_like(array, dtype=np.uint8)

    normalized = (array - min_value) * 255.0 / (max_value - min_value)
    return normalized.astype(np.uint8)


def spatial_entropy(lbp, grid=4, n_bins=10):
    h, w = lbp.shape
    h_step = h // grid
    w_step = w // grid

    entropies = []

    for i in range(grid):
        for j in range(grid):
            patch = lbp[i*h_step:(i+1)*h_step, j*w_step:(j+1)*w_step]

            hist, _ = np.histogram(patch.ravel(), bins=n_bins, range=(0, n_bins))
            hist = hist.astype("float")
            hist /= hist.sum() + 1e-6

            entropies.append(-np.sum(hist * np.log(hist + 1e-6)))

    return np.mean(entropies), np.std(entropies)


def compare_region_histograms(hists):
    dists = []

    for i in range(len(hists)):
        for j in range(i + 1, len(hists)):
            dists.append(cosine(hists[i], hists[j]))

    return np.mean(dists), np.std(dists)


def compute_lbp_visual(gray):
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")

    lbp_norm = _normalize_to_uint8(lbp)
    lbp_color = cv2.applyColorMap(lbp_norm, cv2.COLORMAP_TURBO)

    return lbp, lbp_color


def compute_lbp_image(gray):
    r = 1
    n_points = 8 * r
    n_bins = n_points + 2

    lbp = local_binary_pattern(gray, n_points, r, method="uniform")

    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
    hist = hist.astype("float")
    hist /= hist.sum() + 1e-6

    features = {}

    features["entropy"] = -np.sum(hist * np.log(hist + 1e-6))
    features["uniformity"] = np.sum(hist ** 2)

    ent_mean, ent_std = spatial_entropy(lbp, grid=4, n_bins=n_bins)
    features["entropy_spatial_mean"] = ent_mean
    features["entropy_spatial_std"] = ent_std

    return features, hist


def compute_lbp_frame(frame):
    results = {}
    region_hists = {}

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (256, 256))

    h, w = gray.shape

    regions = {
        "full": gray,
        "top": gray[:h//2, :],
        "bottom": gray[h//2:, :],
        "left": gray[:, :w//2],
        "right": gray[:, w//2:]
    }

    for name, region in regions.items():
        region = cv2.resize(region, (128, 128))

        feats, hist = compute_lbp_image(region)

        for k, v in feats.items():
            results[f"{name}_{k}"] = v

        region_hists[name] = hist

    mean_dist, std_dist = compare_region_histograms(list(region_hists.values()))

    results["lbp_region_distance_mean"] = mean_dist
    results["lbp_region_distance_std"] = std_dist
    results["lbp_vertical_diff"] = cosine(region_hists["top"], region_hists["bottom"])
    results["lbp_horizontal_diff"] = cosine(region_hists["left"], region_hists["right"])

    _, lbp_map_full = compute_lbp_visual(gray)

    return results, lbp_map_full

In [ ]:
def compute_lbp_metrics_video(sample, max_frames=120):
    frames = load_video_frames(sample)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, max_frames).astype(int)
        frames = frames[idx]

    metrics = []

    for frame in frames:
        frame_metrics, _ = compute_lbp_frame(frame)
        metrics.append(frame_metrics)

    return metrics

## Player

In [ ]:
def play_video_lbp_image(sample, interval=40, show_players=True, max_frames=120):
    frames = load_video_frames(sample)

    if len(frames) == 0:
        raise ValueError("Nao ha frames para exibir.")

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, max_frames).astype(int)
        frames = frames[idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")

    _, lbp0 = compute_lbp_frame(frames[0])

    img = ax.imshow(lbp0)

    if show_players:
        def update(i):
            frame = frames[i].copy()

            metrics, lbp_map = compute_lbp_frame(frame)

            texts = [
                f"FULL Entropy: {metrics['full_entropy']:.3f} | Uniformity: {metrics['full_uniformity']:.3f}",
                f"FULL Entropia Espacial: {metrics['full_entropy_spatial_mean']:.3f} / {metrics['full_entropy_spatial_std']:.3f}",
                f"Distancia entre Regioes: {metrics['lbp_region_distance_mean']:.3f} / {metrics['lbp_region_distance_std']:.3f}",
                f"Vertical Diff: {metrics['lbp_vertical_diff']:.3f}",
                f"Horizontal Diff: {metrics['lbp_horizontal_diff']:.3f}"
            ]
            lbp_map = draw_text_block(lbp_map, texts)

            img.set_data(lbp_map)
            return [img]

        ani = animation.FuncAnimation(
            fig,
            update,
            frames=len(frames),
            interval=interval,
            blit=True
        )

        plt.close(fig)
        display(HTML(ani.to_jshtml()))

    return compute_lbp_metrics_video(sample)

In [ ]:
#play_video_lbp_image(metadata_true[1])

In [ ]:
#metrics = play_video_lbp_image(metadata_fake[1])
#metrics 


# Sobel

In [ ]:
import numpy as np
import cv2
from scipy.stats import skew
from scipy.spatial.distance import cosine
from itertools import combinations

In [ ]:
def compute_sobel_forensic(gray):
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)

    mag = cv2.magnitude(gx, gy)
    mag = np.log1p(mag) 

    angle = cv2.phase(gx, gy, angleInDegrees=False)  # radianos direto

    # normalização
    mag_norm = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    mag_norm = mag_norm.astype(np.uint8)

    # colormap forte (visual forense)
    mag_color = cv2.applyColorMap(mag_norm, cv2.COLORMAP_TURBO)

    return mag, angle, mag_color

def sobel_spatial_features(mag, grid=3):
    h, w = mag.shape
    h_bins = np.linspace(0, h, grid+1, dtype=int)
    w_bins = np.linspace(0, w, grid+1, dtype=int)

    patch_means = []
    patch_stds = []

    for i in range(grid):
        for j in range(grid):
            patch = mag[h_bins[i]:h_bins[i+1], w_bins[j]:w_bins[j+1]]

            patch_means.append(np.mean(patch))
            patch_stds.append(np.std(patch))

    patch_means = np.array(patch_means)
    patch_stds = np.array(patch_stds)

    # variação entre regiões
    spatial_mean_std = np.std(patch_means)
    spatial_std_std = np.std(patch_stds)

    return spatial_mean_std, spatial_std_std

def directional_coherence(angle, grid=3):
    h, w = angle.shape
    h_bins = np.linspace(0, h, grid+1, dtype=int)
    w_bins = np.linspace(0, w, grid+1, dtype=int)

    region_vectors = []

    for i in range(grid):
        for j in range(grid):
            patch = angle[h_bins[i]:h_bins[i+1], w_bins[j]:w_bins[j+1]]

            radians = np.deg2rad(patch)

            mean_sin = np.mean(np.sin(radians))
            mean_cos = np.mean(np.cos(radians))

            region_vectors.append([mean_sin, mean_cos])

    region_vectors = np.array(region_vectors)

    std_sin = np.std(region_vectors[:, 0])
    std_cos = np.std(region_vectors[:, 1])

    return std_sin + std_cos

def projection_features(mag, grid=3):
    h, w = mag.shape
    h_bins = np.linspace(0, h, grid+1, dtype=int)
    w_bins = np.linspace(0, w, grid+1, dtype=int)

    grid_vals = np.zeros((grid, grid))

    for i in range(grid):
        for j in range(grid):
            patch = mag[h_bins[i]:h_bins[i+1], w_bins[j]:w_bins[j+1]]
            grid_vals[i, j] = np.mean(patch)

    # média por linha e coluna
    row_means = np.mean(grid_vals, axis=1)
    col_means = np.mean(grid_vals, axis=0)

    horizontal_var = np.std(row_means)
    vertical_var = np.std(col_means)

    return horizontal_var, vertical_var

def compute_sobel_image(gray):
    features = {}

    mag, angle, mag_color = compute_sobel_forensic(gray)

    # intensidade do gradiente
    features["grad_mean"] = np.mean(mag)
    features["grad_std"] = np.std(mag)
    features["grad_p90"] = np.percentile(mag, 90)

    # distribuição de direções
    angle_hist, _ = np.histogram(angle.ravel(), bins=12, range=(0, 360))
    angle_hist = angle_hist.astype("float")
    angle_hist /= angle_hist.sum() + 1e-6

    features["grad_dir_entropy"] = -np.sum(angle_hist * np.log(angle_hist + 1e-6))

    # coerência direcional entre regiões
    features["grad_dir_coherence"] = directional_coherence(angle, grid=3)

    # estatísticas espaciais
    spatial_mean_std, spatial_std_std = sobel_spatial_features(mag, grid=3)

    features["grad_spatial_mean_std"] = spatial_mean_std
    features["grad_spatial_std_std"] = spatial_std_std

    # estatísticas de projeção, estrutura global
    h_var, v_var = projection_features(mag, grid=3)

    features["grad_horizontal_var"] = h_var
    features["grad_vertical_var"] = v_var

    return features, angle_hist, mag_color

def compute_semantic_distances(region_angle_hists):
    results = {}

    def safe_cosine(a, b):
        return cosine(a, b) if a is not None and b is not None else 0

    # frame interno
    if "upper_frame" in region_angle_hists and "lower_frame" in region_angle_hists:
        results["frame_internal_angle_dist"] = safe_cosine(
            region_angle_hists["upper_frame"],
            region_angle_hists["lower_frame"]
        )
    if "left_frame" in region_angle_hists and "right_frame" in region_angle_hists:
        results["frame_lr_angle_dist"] = safe_cosine(
            region_angle_hists["left_frame"],
            region_angle_hists["right_frame"]
        )

    return results

def region_heterogeneity(metrics):
    regions = ["upper_frame", "lower_frame", "left_frame", "right_frame"]

    values = [
        metrics[f"{r}_grad_std"]
        for r in regions
        if f"{r}_grad_std" in metrics
    ]

    if len(values) < 2:
        return {}

    values = np.array(values)

    return {
        "sobel_region_std": np.std(values),              # dispersão global
        "sobel_region_range": np.max(values) - np.min(values)  # max-min (ranking)
    }

def directional_asymmetry(metrics):
    results = {}

    if "upper_frame_grad_dir_coherence" in metrics and "lower_frame_grad_dir_coherence" in metrics:
        results["sobel_tb_coherence_diff"] = abs(
            metrics["upper_frame_grad_dir_coherence"] -
            metrics["lower_frame_grad_dir_coherence"]
        )

    if "left_frame_grad_dir_coherence" in metrics and "right_frame_grad_dir_coherence" in metrics:
        results["sobel_lr_coherence_diff"] = abs(
            metrics["left_frame_grad_dir_coherence"] -
            metrics["right_frame_grad_dir_coherence"]
        )

    return results

def angular_consistency(region_angle_hists):
    dists = []

    for a, b in combinations(region_angle_hists.keys(), 2):
        dists.append(cosine(region_angle_hists[a], region_angle_hists[b]))

    if len(dists) == 0:
        return {}

    dists = np.array(dists)

    return {
        "sobel_angle_consistency_mean": np.mean(dists),
        "sobel_angle_consistency_std": np.std(dists),
    }

def sobel_score_frame(metrics):

    # heterogeneidade entre regiões, mede a falta de coerência local, quanto mais heterogêneo, mais provável de ser fake
    heterogeneity = (
        metrics.get("sobel_region_std", 0) +
        metrics.get("sobel_region_range", 0)
    )

    # assimetria direcional entre regiões, mede a falta de coerência local, quanto mais assimétrico, mais provável de ser fake
    asymmetry = (
        metrics.get("sobel_tb_coherence_diff", 0) +
        metrics.get("sobel_lr_coherence_diff", 0)
    )

    # inconsistência angular entre regiões, mede a falta de coerência global, quanto mais inconsistente, mais provável de ser fake
    angular = (
        metrics.get("sobel_angle_consistency_mean", 0) * 2 +
        metrics.get("sobel_angle_consistency_std", 0) +
        metrics.get("frame_internal_angle_dist", 0) +
        metrics.get("frame_lr_angle_dist", 0)
    )

    # score final ponderado
    score = (
        0.4 * heterogeneity +
        0.3 * asymmetry +
        0.3 * angular
    )

    return score

In [ ]:
def compute_sobel_frame(frame):

    results = {}
    region_angle_hists = {}

    gray_full = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    h, w = gray_full.shape

    # regiões: frame inteiro, metade superior, metade inferior, metade esquerda, metade direita
    all_regions = {
        "full_frame": gray_full,
        "upper_frame": gray_full[:gray_full.shape[0]//2, :],
        "lower_frame": gray_full[gray_full.shape[0]//2:, :],
        "left_frame": gray_full[:, :gray_full.shape[1]//2],
        "right_frame": gray_full[:, gray_full.shape[1]//2:]
    }

    # extrair features de cada região
    for region_name, gray in all_regions.items():

        if gray.shape[0] < 32 or gray.shape[1] < 32:
            continue

        features, angle_hist, _ = compute_sobel_image(gray)

        for k, v in features.items():
            results[f"{region_name}_{k}"] = v

        region_angle_hists[region_name] = angle_hist

    # comparação entre regiões
    results.update(compute_semantic_distances(region_angle_hists))
    # mede a heterogeneidade entre as regiões, o quanto elas são diferentes em termos de textura, quanto maior, mais heterogêneo é o frame
    results.update(region_heterogeneity(results))
    # mede a assimetria direcional entre as regiões, o quanto as direções dos gradientes são diferentes entre as regiões opostas, quanto maior, mais assimétrico é o frame
    results.update(directional_asymmetry(results))
    # mede a consistência angular entre as regiões, o quanto as direções dos gradientes são similares entre as regiões, quanto menor, mais consistente é o frame
    results.update(angular_consistency(region_angle_hists))

    #score
    results["sobel_score"] = sobel_score_frame(results)

    # mapa visual do frame inteiro
    _, _, sobel_map_full = compute_sobel_forensic(gray_full)

    return results, sobel_map_full

In [ ]:
def sobel_score_video(scores):

    scores = np.array(scores)

    mean_score = np.mean(scores)
    std_score = np.std(scores)
    p90_score = np.percentile(scores, 90)
    instability = np.mean(np.abs(np.diff(scores))) if len(scores) > 1 else 0

    final_score = (
        0.35 * mean_score +
        0.25 * p90_score +
        0.20 * std_score +
        0.20 * instability
    )

    return {
        "sobel_final_score": final_score,
        "mean": mean_score,
        "std": std_score,
        "p90": p90_score,
        "instability": instability
    }

In [ ]:
def compute_sobel_scores(sample):
    frames = load_video_frames(sample)

    frame_scores = []
    frame_metrics = []

    for frame in frames:

        metrics, _ = compute_sobel_frame(frame)

        score = sobel_score_frame(metrics)

        frame_scores.append(score)
        frame_metrics.append(metrics)

    scores = np.array(frame_scores)
    scores = (scores - np.mean(scores)) / (np.std(scores) + 1e-6)

    video_result = sobel_score_video(frame_scores)

    return {
        "final_score": video_result,
        "frame_scores": scores,
        "frame_metrics": frame_metrics
    }

## Player

In [ ]:
def play_video_sobel_image(sample, interval=40, show_players=True, max_frames=2000):
    frames = load_video_frames(sample)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, max_frames).astype(int)
        frames = frames[idx]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.axis("off")

    _, sobel0 = compute_sobel_frame(frames[0])

    img = ax.imshow(sobel0)

    if show_players:
        def update(i):
            frame = frames[i].copy()

            metrics, sobel_map = compute_sobel_frame(frame)

            # overlay de texto
            texts = [
                "[INTENSITY]",
                f"Mean: {metrics['full_frame_grad_mean']:.3f}",
                f"Std: {metrics['full_frame_grad_std']:.3f}",
                f"P90: {metrics['full_frame_grad_p90']:.3f}",

                "",

                "[DIRECTION]",
                f"Entropy: {metrics['full_frame_grad_dir_entropy']:.3f}",
                f"Coherence: {metrics['full_frame_grad_dir_coherence']:.3f}",

                "",

                "[SPATIAL]",
                f"Mean STD: {metrics['full_frame_grad_spatial_mean_std']:.3f}",
                f"STD STD: {metrics['full_frame_grad_spatial_std_std']:.3f}",

                "",

                "[STRUCTURE]",
                f"Horizontal-Var: {metrics['full_frame_grad_horizontal_var']:.3f}",
                f"Vertical-Var: {metrics['full_frame_grad_vertical_var']:.3f}",

                "",

                "[CONSISTENCY]",
                f"Top-Bottom: {metrics.get('frame_internal_angle_dist', 0):.5f}",
                f"Left-Right: {metrics.get('frame_lr_angle_dist', 0):.5f}",

                "[SCORE]",
                f"Frame Score: {metrics['sobel_score']:.3f}"
            ]
            sobel_map = draw_text_block(sobel_map, texts)

            img.set_data(sobel_map)
            return [img]

        ani = animation.FuncAnimation(
            fig,
            update,
            frames=len(frames),
            interval=interval,
            blit=True
        )

        plt.close(fig)
        plt.close(fig)

        import matplotlib as mpl
        mpl.rcParams["animation.embed_limit"] = 500000000

        display(HTML(ani.to_jshtml()))

    return compute_sobel_scores(sample)

In [ ]:
metrics_fake = play_video_sobel_image(metadata_true.iloc[0]['video_path'])

In [ ]:
metrics_fake

# Laplacian

In [ ]:
def laplacian_var(gray):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()


def compute_laplacian_temporal(sample,  interval=30):
    frames = load_video_frames(sample)
    indices = np.linspace(0, len(frames) - 1, min(len(frames), 120)).astype(int)

    values = []
    video_frames = []
    video_metrics = []
    prev_value = None

    for idx in indices:
        gray = rgb_to_gray(sample, idx)
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        val = lap.var()
        values.append(val)

        delta = 0.0 if prev_value is None else abs(val - prev_value)
        prev_value = val

        if show_player:
            lap_img = cv2.normalize(np.abs(lap), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            lap_bgr = cv2.cvtColor(lap_img, cv2.COLOR_GRAY2BGR)
            video_frames.append(lap_bgr)
            video_metrics.append(
                f"Frame: {idx}\n"
                f"Variance: {val:.4f}\n"
                f"Delta: {delta:.4f}"
            )

    results = {
        "laplacian_mean": np.mean(values),
        "laplacian_var_temporal": np.var(values),
        "laplacian_mean_delta": np.mean(np.abs(np.diff(values))),
    }

    return results